In [2]:
import matplotlib.pyplot as plt
import numpy as np

from astropy.visualization import time_support
from astropy.time import Time
import astropy.units as u

from sunpy import timeseries as ts
from sunpy.net import Fido
from sunpy.net import attrs as a

from stixpy.net.client import STIXClient
from stixpy.timeseries import quicklook 
from stixpy.product import Product

import datetime as dt
from sunpy.time import parse_time
from sunpy.time import TimeRange

import pandas as pd

from scipy.signal import find_peaks, savgol_filter

In [3]:
flares = pd.read_csv("C:/Users/derva/OneDrive/Documents/DIAS/solar-orbiter-flare-statistics/data/STIX_flarelist_w_locations_20210101_20260130_version1_python.csv")
top100 = flares.sort_values("goes_estimated_mean_flux", ascending=False).head(100)

In [4]:
def t(string):
    return parse_time(string).datetime

def get_energy_indices(qtable, ranges):
    energy_indices = []
    for e_min, e_max in ranges:
        start_index = np.where(qtable["e_low"] >= e_min)[0][0]
        end_index = np.where(qtable["e_high"] <= e_max)[0][-1]
        energy_indices.append([int(start_index), int(end_index)])
    return energy_indices
    
def get_stix_df(stix_sci, energy_ranges):
    
    energy_indices = get_energy_indices(stix_sci.energies, energy_ranges)
    counts, errors, times, timedeltas, energies = stix_sci.get_data(detector_indices=[[0, 31]],
                                                                    pixel_indices=[[0, 11]],
                                                                    energy_indices=energy_indices,)
    counts = counts.to(u.ct / u.s / u.keV)
    errors = errors.to(u.ct / u.s / u.keV)
    timedeltas = timedeltas.to(u.s)
    
    times = times
    
    energy_columns = [f"{e['e_low']}-{e['e_high']}" for e in energies]
    
    counts_reshaped = counts[:, 0, 0, :]
    
    counts_df = pd.DataFrame(counts_reshaped, index=times.datetime, columns=energy_columns)
    return counts_df

In [5]:
energy_ranges = [(25.0*u.keV, 50*u.keV), (50.0*u.keV, 100*u.keV)]

In [6]:
def rank_overlaps(start, end, sci_query):
    overlaps=[]
    for n in range(len(sci_query[0])):
        latest_start = max(start, sci_query[0][n][0].datetime)
        earliest_end = min(end, sci_query[0][n][1].datetime)
        
        overlap = earliest_end - latest_start
        overlaps.append((overlap.total_seconds(), n))
        
    return np.array(sorted(overlaps, key=lambda x: x[0], reverse=True))

In [ ]:
df_list=[]
indices=[]
for ind in top100.index:
    sci_query = Fido.search(a.Time(top100['start_UTC'].loc[ind], 
                                   top100['end_UTC'].loc[ind]), 
                        a.Instrument.stix,
                        a.stix.DataType.sci,
                        a.stix.DataProduct.sci_xray_spec)
    sci_query['stix'].filter_for_latest_version()

    start = parse_time(top100['start_UTC'].loc[ind]).datetime
    end = parse_time(top100['end_UTC'].loc[ind]).datetime

    overlaps_ranked = rank_overlaps(start, end, sci_query)
    
    for index in overlaps_ranked[:,1]:
        badrange=False
        sci_files = Fido.fetch(sci_query[0][int(index)])
        sci_data = Product(sci_files)
    
        if sci_data.energies["e_high"][len(sci_data.energies["e_high"])-1]<100*u.keV or sci_data.energies["e_low"][0]>25*u.keV:
            badrange=True
        
        if not badrange:
            sci_df = get_stix_df(sci_data, energy_ranges)
            df_list.append(sci_df)
            indices.append(ind)
            break

Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

...

Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

In [ ]:
indices_array = np.array(indices)
incomplete_file_ind = [5, 18, 26, 29]

In [ ]:
for ind in incomplete_file_ind:
    sci_query = Fido.search(a.Time(top100['start_UTC'].iloc[ind], top100['end_UTC'].iloc[ind]), 
                        a.Instrument.stix,
                        a.stix.DataType.sci,
                        a.stix.DataProduct.sci_xray_spec)
    sci_query['stix'].filter_for_latest_version()
    sci_query

    sci_files = Fido.fetch(sci_query)
    sci_files = sorted(sci_files)
    
    
    sci_data_A = Product(sci_files[0])
    sci_data_B = Product(sci_files[1])
    
    df_A = get_stix_df(sci_data_A, energy_ranges)
    df_B = get_stix_df(sci_data_B, energy_ranges)
    
    combined_df = pd.concat([df_A, df_B])
    
    combined_df = combined_df.sort_index()
    
    df = combined_df[~combined_df.index.duplicated(keep="first")]

    df_list[ind]=df

In [ ]:
exclude = [26, 33, 35, 36, 55, 56, 58, 59, 61, 66, 70, 73, 77, 80, 83, 90, 96]

In [ ]:
df_list_refined = df_list.copy()
for ind in indices:
    if ind in exclude:
        df_list_refined[ind]='excluded'

indices_refined = list(set(indices)-set(top100.index[exclude]))

In [ ]:
df_list_filtered = []
for loc, ind in enumerate(indices):
    if ind in indices_refined:
        df_filtered  = df_list[loc].copy()
        
        filtered_2550 = savgol_filter(df_list[loc]['25.0 keV-50.0 keV'], window_length = 25, polyorder=2)
        filtered_50100 = savgol_filter(df_list[loc]['50.0 keV-100.0 keV'], window_length = 25, polyorder=2)

        df_filtered['25.0 keV-50.0 keV'] = filtered_2550
        df_filtered['50.0 keV-100.0 keV'] = filtered_50100

        df_list_filtered.append(df_filtered)
        
    if ind not in indices_refined:
        df_list_filtered.append('excluded')
        

In [ ]:
columns = ('flare_list_id', 'start_time', 'end_time','peak_time', 'peak_counts', 'number_of_peaks', 'rise_time', 'decay_time','start_time (50-100 keV)', 'end_time (50-100 keV)', 'peak_counts (50-100 keV)', 'rise_time (50-100 keV)', 'decay_time (50-100 keV)', 'FWHM (25-50 keV)', 'FWHM (50-100 keV)', 'comments')
data=np.zeros((len(indices_refined), len(columns)))


In [ ]:
filtered_stats_df = pd.DataFrame(data, columns = columns)

In [ ]:
filtered_stats_df['flare_list_id'] = filtered_stats_df['flare_list_id'].astype(int)
label = 0
for loc, ind in enumerate(indices):
    if loc not in exclude:
        filtered_stats_df.loc[label, 'flare_list_id'] = int(ind)
        label+=1

In [ ]:
filtered_stats_df['start_time'] = filtered_stats_df['start_time'].astype(str)
filtered_stats_df['end_time'] = filtered_stats_df['end_time'].astype(str)
filtered_stats_df['rise_time'] = filtered_stats_df['rise_time'].astype(str)
filtered_stats_df['decay_time'] = filtered_stats_df['decay_time'].astype(str)
filtered_stats_df['peak_time'] = filtered_stats_df['peak_time'].astype(str)
filtered_stats_df['start_time (50-100 keV)'] = filtered_stats_df['start_time (50-100 keV)'].astype(str)
filtered_stats_df['end_time (50-100 keV)'] = filtered_stats_df['end_time (50-100 keV)'].astype(str)
filtered_stats_df['rise_time (50-100 keV)'] = filtered_stats_df['rise_time (50-100 keV)'].astype(str)
filtered_stats_df['decay_time (50-100 keV)'] = filtered_stats_df['decay_time (50-100 keV)'].astype(str)

In [ ]:
label=0
i=0
for ind in top100.index:
    if i not in exclude:
        sci_df = df_list_filtered[np.where(indices_array==ind)[0][0]]
    
        start = parse_time(top100['start_UTC'].loc[ind]).datetime - dt.timedelta(minutes=30)
        end = parse_time(top100['end_UTC'].loc[ind]).datetime  + dt.timedelta(minutes=30)
    
        plot_df = sci_df.truncate(start, end)
        plot_df.index = parse_time(plot_df.index).datetime 
        
        r = top100['solo_position_AU_distance'].loc[ind]

        peak_counts = np.max(plot_df['25.0 keV-50.0 keV'])
        peak_counts_scaled = peak_counts/r**2
        filtered_stats_df.loc[label, 'peak_counts'] = peak_counts_scaled
        peak_loc = plot_df.index[plot_df['25.0 keV-50.0 keV'] == peak_counts][0]
        filtered_stats_df.loc[label, 'peak_time'] = str(peak_loc)

        flare = plot_df[plot_df['25.0 keV-50.0 keV']>10/r**2]
        tstart = flare.index[0]
        tend = plot_df[(plot_df['25.0 keV-50.0 keV']<10/r**2) & (plot_df.index>peak_loc)].index[0]

        filtered_stats_df.loc[label, 'start_time'] = str(tstart)
        filtered_stats_df.loc[label, 'end_time'] = str(tend)
        
        plot_df = plot_df.truncate(t(tstart), t(tend))

        rise_time = peak_loc - tstart
        decay_time = tend - peak_loc
        filtered_stats_df.loc[label, 'rise_time'] = str(rise_time).split(" ")[-1]
        filtered_stats_df.loc[label, 'decay_time'] = str(decay_time).split(" ")[-1]

        peak_counts_50_100 = np.max(plot_df['50.0 keV-100.0 keV'])
        peak_counts_50_100_scaled = peak_counts_50_100/r**2
        filtered_stats_df.loc[label, 'peak_counts (50-100 keV)'] = peak_counts_50_100_scaled
        peak_loc_50_100 = plot_df.index[plot_df['50.0 keV-100.0 keV'] == peak_counts_50_100][0]

        flare_50_100 = plot_df[plot_df['50.0 keV-100.0 keV']>3/r**2]

        if flare_50_100.empty:
            filtered_stats_df.loc[label, 'rise_time (50-100 keV)'] = np.nan
            filtered_stats_df.loc[label, 'decay_time (50-100 keV)'] = np.nan
            filtered_stats_df.loc[label, 'start_time (50-100 keV)'] = np.nan
            filtered_stats_df.loc[label, 'end_time (50-100 keV)'] = np.nan
        
        else:
            tstart_50_100 = flare_50_100.index[0]
            #tend_50_100 = flare_50_100.index[-1]
            # tend_50_100 = plot_df[(plot_df['50.0 keV-100.0 keV']<3/r**2) & (plot_df.index>peak_loc_50_100)]
            # if tend_50_100.empty:
            #     findend = plot_df[(plot_df.index>peak_loc_50_100)&(plot_df.index<t(tend))]
            #     tend_50_100 = findend.index[findend['50.0 keV-100.0 keV'] == np.min(findend['50.0 keV-100.0 keV'])][-1]
            # else:
                #tend_50_100 = tend_50_100.index[0]
            tend_50_100 = flare_50_100.index[-1]

            rise_time_50_100 = peak_loc_50_100 - tstart_50_100
            decay_time_50_100 = tend_50_100 - peak_loc_50_100
            filtered_stats_df.loc[label, 'rise_time (50-100 keV)'] = str(rise_time_50_100).split(" ")[-1]
            filtered_stats_df.loc[label, 'decay_time (50-100 keV)'] = str(decay_time_50_100).split(" ")[-1]
            filtered_stats_df.loc[label, 'start_time (50-100 keV)'] = str(tstart_50_100)
            filtered_stats_df.loc[label, 'end_time (50-100 keV)'] = str(tend_50_100)

        peaks, properties = find_peaks(plot_df['25.0 keV-50.0 keV'], distance=30, prominence=0.1*peak_counts)
        filtered_stats_df.loc[label, 'number_of_peaks'] = len(peaks)
        #peak_locs[label]=peaks

        label+=1
    i+=1

In [ ]:
i=0
peak_locs = [[] for _ in range(len(indices_refined))]
for ind in indices:
    if ind in indices_refined:
        fig, ax = plt.subplots()
        ax.set_ylabel('ct/(keV s)')
        ax.set_title(f"{parse_time(top100['start_UTC'].loc[ind]).datetime.year}-{parse_time(top100['start_UTC'].loc[ind]).datetime.month}")
        sci_df = df_list_filtered[np.where(indices_array==ind)[0][0]]

        r = top100['solo_position_AU_distance'].loc[ind]
        
        tstart = filtered_stats_df['start_time'].iloc[i]
        tend = filtered_stats_df['end_time'].iloc[i]

        tstart_50_100 = filtered_stats_df['start_time (50-100 keV)'].iloc[i]
        tend_50_100 = filtered_stats_df['end_time (50-100 keV)'].iloc[i]

        plt.axvline(t(tend), 0, 1, color = 'palevioletred', linestyle = 'dashed', label = 'Estimated Start/End (25-50 keV)')
        plt.axvline(t(tstart), 0, 1, color = 'palevioletred', linestyle = 'dashed')
        plt.axhline(10/r**2, 0, 1, color='red', linestyle='dotted', label='10 counts/keV s (scaled)')
        plt.axhline(3/r**2, 0, 1, color='salmon', linestyle='dotted', label='3 counts/keV s (scaled)')

        if pd.notna(tend_50_100):
            plt.axvline(t(tend_50_100), 0, 1, color = 'pink', linestyle = 'dashed', label = 'Estimated Start/End (50-100 keV)')
            plt.axvline(t(tstart_50_100), 0, 1, color = 'pink', linestyle = 'dashed')

        rise_time_td = pd.to_timedelta(filtered_stats_df['rise_time'].iloc[i])
        decay_time_td = pd.to_timedelta(filtered_stats_df['decay_time'].iloc[i])
        
        duration = rise_time_td+decay_time_td
        duration = duration.total_seconds()
        
        plot_df = sci_df.truncate(t(tstart)-dt.timedelta(seconds=0.1*duration), t(tend)+dt.timedelta(seconds=0.1*duration))
        for n in range(len(plot_df.columns)):
            ax.plot(plot_df.index, plot_df[plot_df.columns[n]], label=plot_df.columns[n])
        
        handles, labels = ax.get_legend_handles_labels()
        fig.legend(handles, labels, loc='upper right', fontsize='small')
        fig.savefig("C:/Users/derva/OneDrive/Documents/DIAS/solar-orbiter-flare-statistics/figures/week_4/top_100_filtered_cropped/top_100_flares_science_data_plot"+str(i)+"_filtered_cropped.png", bbox_inches='tight')

        i+=1
    
plt.close('all')

In [ ]:
rise_time_td = pd.to_timedelta(filtered_stats_df['rise_time'])
decay_time_td = pd.to_timedelta(filtered_stats_df['decay_time'])

durations = rise_time_td+decay_time_td
durations = durations.dt.total_seconds()

In [ ]:
np.where(np.array(durations)>2000)

In [ ]:
plt.hist(durations, bins=50)
plt.xlabel('Flare duration (seconds)')
plt.title('25-50 keV')
plt.savefig("C:/Users/derva/OneDrive/Documents/DIAS/solar-orbiter-flare-statistics/figures/week_4/top_100_filtered/duration_hist_filtered.png", bbox_inches='tight')

In [ ]:
rise_time_50_100_td = pd.to_timedelta(filtered_stats_df['rise_time (50-100 keV)'])
decay_time_50_100_td = pd.to_timedelta(filtered_stats_df['decay_time (50-100 keV)'])

durations_50_100 = rise_time_50_100_td+decay_time_50_100_td
durations_50_100 = durations_50_100.dt.total_seconds()

In [ ]:
plt.hist(durations_50_100, bins=50)
plt.xlabel('Flare duration (seconds)')
plt.title('50-100 keV')
plt.savefig("C:/Users/derva/OneDrive/Documents/DIAS/solar-orbiter-flare-statistics/figures/week_4/top_100_filtered/duration_hist_50_100_filtered.png", bbox_inches='tight')

In [ ]:
plt.scatter(filtered_stats_df['peak_counts'], durations)
plt.yscale('log')
plt.xscale('log')
plt.ylabel('Flare duration (seconds)')
plt.title('25-50 keV')
plt.xlabel('Scaled peak counts')
#plt.savefig("C:/Users/derva/OneDrive/Documents/DIAS/solar-orbiter-flare-statistics/figures/week_4/top_100_filtered/duration_vs_peak_counts_filtered.png", bbox_inches='tight')

In [ ]:
plt.scatter(filtered_stats_df['peak_counts (50-100 keV)'], durations_50_100)
plt.yscale('log')
plt.xscale('log')
plt.ylabel('Flare duration (seconds)')
plt.title('50-100 keV')
plt.xlabel('Scaled peak counts')
#plt.savefig("C:/Users/derva/OneDrive/Documents/DIAS/solar-orbiter-flare-statistics/figures/week_4/top_100_filtered/duration_vs_peak_counts_50_100_filtered.png", bbox_inches='tight')

In [ ]:
peak_ratios = filtered_stats_df['peak_counts (50-100 keV)']/filtered_stats_df['peak_counts']
plt.hist(peak_ratios, bins=50)
plt.xlabel('Ratio of peaks (50-100 keV:25-50 keV)')

In [ ]:
rise_time_td

In [ ]:
rise_time_s = rise_time_td.dt.total_seconds()
decay_time_s = decay_time_td.dt.total_seconds()

rise_decay_ratio = rise_time_s/decay_time_s

rise_time_s_50100 = rise_time_50_100_td.dt.total_seconds()
decay_time_s_50100 = decay_time_50_100_td.dt.total_seconds()

rise_decay_ratio_50100 = rise_time_s_50100/decay_time_s_50100

plt.hist(rise_decay_ratio, bins=50)

In [ ]:
plt.hist(rise_decay_ratio_50100, bins=50)

In [ ]:
plt.scatter(peak_ratios, durations)
plt.yscale('log')
plt.xlabel('Peak Ratio (50-100 keV:25-50 keV)')
plt.ylabel('Duration (seconds)')
plt.title('25-50 keV')
plt.savefig("C:/Users/derva/OneDrive/Documents/DIAS/solar-orbiter-flare-statistics/figures/week_4/top_100_filtered/duration_vs_peak_ratio.png", bbox_inches='tight')

In [ ]:
plt.scatter(peak_ratios, durations_50_100)
plt.yscale('log')
plt.xlabel('Peak Ratio (50-100 keV:25-50 keV)')
plt.ylabel('Duration (seconds)')
plt.title('50-100 keV')
plt.savefig("C:/Users/derva/OneDrive/Documents/DIAS/solar-orbiter-flare-statistics/figures/week_4/top_100_filtered/duration_vs_peak_ratio_50_100.png", bbox_inches='tight')

In [ ]:
i=0
peak_locs = [[] for _ in range(len(indices_refined))]
for ind in indices:
    if ind in indices_refined:
        fig, ax = plt.subplots()
        ax.set_ylabel('ct/(keV s)')
        ax.set_title(f"{parse_time(top100['start_UTC'].loc[ind]).datetime.year}-{parse_time(top100['start_UTC'].loc[ind]).datetime.month}")
        sci_df = df_list[np.where(indices_array==ind)[0][0]]

        r = top100['solo_position_AU_distance'].loc[ind]
        
        tstart = filtered_stats_df['start_time'].iloc[i]
        tend = filtered_stats_df['end_time'].iloc[i]

        tstart_50_100 = filtered_stats_df['start_time (50-100 keV)'].iloc[i]
        tend_50_100 = filtered_stats_df['end_time (50-100 keV)'].iloc[i]

        plt.axvline(t(tend), 0, 1, color = 'palevioletred', linestyle = 'dashed', label = 'Estimated Start/End (25-50 keV)')
        plt.axvline(t(tstart), 0, 1, color = 'palevioletred', linestyle = 'dashed')
        plt.axhline(10/r**2, 0, 1, color='red', linestyle='dotted', label='10 counts/keV s (scaled)')
        plt.axhline(3/r**2, 0, 1, color='salmon', linestyle='dotted', label='3 counts/keV s (scaled)')

        if pd.notna(tend_50_100):
            plt.axvline(t(tend_50_100), 0, 1, color = 'pink', linestyle = 'dashed', label = 'Estimated Start/End (50-100 keV)')
            plt.axvline(t(tstart_50_100), 0, 1, color = 'pink', linestyle = 'dashed')
        
        plot_df = sci_df.truncate(t(tstart)-dt.timedelta(minutes=10), t(tend)+dt.timedelta(minutes=10))
        for n in range(len(plot_df.columns)):
            ax.plot(plot_df.index, plot_df[plot_df.columns[n]], label=plot_df.columns[n])
        
        handles, labels = ax.get_legend_handles_labels()
        fig.legend(handles, labels, loc='upper right', fontsize='small')
        fig.savefig("C:/Users/derva/OneDrive/Documents/DIAS/solar-orbiter-flare-statistics/figures/week_4/top_100_flares_refined/top_100_flares_science_data_plot"+str(i)+".png", bbox_inches='tight')

        i+=1
    
plt.close('all')

In [ ]:
label=0
i=0
half_max_points = []
half_max_vals = []
end_points = []
end_point_vals = []
fwhm_list = []
half_max_points_50100 = []
half_max_vals_50100 = []
end_points_50100 = []
end_point_vals_50100 = []
fwhm_list_50100 = []
for ind in top100.index:
    if i not in exclude:
        sci_df = df_list_filtered[np.where(indices_array==ind)[0][0]]
    
        tstart = filtered_stats_df['start_time'].iloc[label]
        tend = filtered_stats_df['end_time'].iloc[label]
        tpeak = filtered_stats_df['peak_time'].iloc[label]

        r = top100['solo_position_AU_distance'].loc[ind]
        max_counts = (filtered_stats_df['peak_counts'].iloc[label])*r**2

        plot_df = sci_df.truncate(t(tstart)-dt.timedelta(minutes=10), t(tend)+dt.timedelta(minutes=10))

        half_max = 0.5*max_counts
        half_max_loc = plot_df.index[plot_df['25.0 keV-50.0 keV'] >= half_max][0]
        half_max_points.append(half_max_loc)
        half_max_vals.append(half_max)

        over_half = np.where(plot_df['25.0 keV-50.0 keV']>half_max)
        end_point = plot_df.index[over_half[0][-1]]
        end_points.append(end_point)
        end_point_vals.append(plot_df['25.0 keV-50.0 keV'].iloc[over_half[0][-1]])

        # rise_time_50_100_td = pd.to_timedelta(filtered_stats_df['rise_time (50-100 keV)'])
        # decay_time_50_100_td = pd.to_timedelta(filtered_stats_df['decay_time (50-100 keV)'])
        
        # durations_50_100 = rise_time_50_100_td+decay_time_50_100_td
        # durations_50_100 = durations_50_100.dt.total_seconds()

        fwhm = (t(end_point)-t(half_max_loc))
            #.total_seconds()
        fwhm_list.append(fwhm)

        filtered_stats_df.loc[label, 'FWHM (25-50 keV)'] = fwhm.total_seconds()

        if pd.notna(filtered_stats_df['start_time (50-100 keV)'].iloc[label]):
            max_counts_50100 = (filtered_stats_df['peak_counts (50-100 keV)'].iloc[label])*r**2
    
            half_max_50100 = 0.5*max_counts_50100
            half_max_loc_50100 = plot_df.index[plot_df['50.0 keV-100.0 keV'] >= half_max_50100][0]
            half_max_points_50100.append(half_max_loc_50100)
            half_max_vals_50100.append(half_max_50100)
    
            over_half_50100 = np.where(plot_df['50.0 keV-100.0 keV']>half_max_50100)
            end_point_50100 = plot_df.index[over_half_50100[0][-1]]
            end_points_50100.append(end_point_50100)
            end_point_vals_50100.append(plot_df['50.0 keV-100.0 keV'].iloc[over_half_50100[0][-1]])
            
            fwhm_50100 = (t(end_point_50100)-t(half_max_loc_50100))
                #.total_seconds()
            fwhm_list_50100.append(fwhm_50100)

            filtered_stats_df.loc[label, 'FWHM (50-100 keV)'] = fwhm_50100.total_seconds()

        else:
            filtered_stats_df.loc[label, 'FWHM (50-100 keV)'] = np.nan
            fwhm_list_50100.append(np.nan)
            half_max_vals_50100.append(np.nan)
            half_max_points_50100.append(np.nan)
            end_points_50100.append(np.nan)
            end_point_vals_50100.append(np.nan)
            
        label+=1
    i+=1

In [ ]:
#half_max_points_50100

In [ ]:
t(half_max_loc) + fwhm

In [ ]:
i=0
peak_locs = [[] for _ in range(len(indices_refined))]
for ind in indices:
    if ind in indices_refined:
        fig, ax = plt.subplots()
        ax.set_ylabel('ct/(keV s)')
        ax.set_title(f"{parse_time(top100['start_UTC'].loc[ind]).datetime.year}-{parse_time(top100['start_UTC'].loc[ind]).datetime.month}")
        sci_df = df_list_filtered[np.where(indices_array==ind)[0][0]]

        r = top100['solo_position_AU_distance'].loc[ind]
        
        tstart = filtered_stats_df['start_time'].iloc[i]
        tend = filtered_stats_df['end_time'].iloc[i]

        tstart_50_100 = filtered_stats_df['start_time (50-100 keV)'].iloc[i]
        tend_50_100 = filtered_stats_df['end_time (50-100 keV)'].iloc[i]

        plt.axvline(t(tend), 0, 1, color = 'palevioletred', linestyle = 'dashed', label = 'Estimated Start/End (25-50 keV)')
        plt.axvline(t(tstart), 0, 1, color = 'palevioletred', linestyle = 'dashed')
        
        plot_df = sci_df.truncate(t(tstart)-dt.timedelta(minutes=10), t(tend)+dt.timedelta(minutes=10))
        for n in range(len(plot_df.columns)):
            ax.plot(plot_df.index, plot_df[plot_df.columns[n]], label=plot_df.columns[n])

        ax.plot(t(half_max_points[i]), half_max_vals[i], marker='o',ms=3, color='lightseagreen')
        ax.hlines(half_max_vals[i], t(half_max_points[i]), t(half_max_points[i]) + fwhm_list[i], linestyle='dashed', color='lightseagreen')
        ax.plot(t(end_points[i]), end_point_vals[i], marker='o', ms=3, color='lightseagreen')

        if pd.notna(fwhm_list_50100[i]):
            ax.plot(t(half_max_points_50100[i]), half_max_vals_50100[i], marker='o', ms=3, color='lightseagreen')
            ax.hlines(half_max_vals_50100[i], t(half_max_points_50100[i]), t(half_max_points_50100[i]) + fwhm_list_50100[i], linestyle='dashed', color='lightseagreen')
            ax.plot(t(end_points_50100[i]), end_point_vals_50100[i], marker='o', ms=3, color='lightseagreen')
        
        handles, labels = ax.get_legend_handles_labels()
        fig.legend(handles, labels, loc='upper right', fontsize='small')
        fig.savefig("C:/Users/derva/OneDrive/Documents/DIAS/solar-orbiter-flare-statistics/figures/week_4/fwhm_testing/flare"+str(i)+".png", bbox_inches='tight')

        i+=1
    
plt.close('all')

In [ ]:
plt.hist(filtered_stats_df['FWHM (25-50 keV)'], bins=50)
plt.xlabel('Approximate FWHM')
plt.title('25-50 keV')
plt.savefig("C:/Users/derva/OneDrive/Documents/DIAS/solar-orbiter-flare-statistics/figures/week_4/fwhm_testing/fwhm_hist.png", bbox_inches='tight')

In [ ]:
plt.hist(filtered_stats_df['FWHM (50-100 keV)'], bins=50)
plt.xlabel('Approximate FWHM')
plt.title('50-100 keV')
plt.savefig("C:/Users/derva/OneDrive/Documents/DIAS/solar-orbiter-flare-statistics/figures/week_4/fwhm_testing/fwhm_50_100_hist.png", bbox_inches='tight')

In [ ]:
tstart = filtered_stats_df['start_time'].iloc[0]
tend = filtered_stats_df['end_time'].iloc[0]

plot_df= df_list_filtered[0].truncate(t(tstart)-dt.timedelta(minutes=10), t(tend)+dt.timedelta(minutes=10))
plt.plot(plot_df.index, plot_df['25.0 keV-50.0 keV'].diff())

In [ ]:
test_df = plot_df[plot_df['25.0 keV-50.0 keV'].diff()>0.1]


In [ ]:
plt.plot(test_df.index, test_df['25.0 keV-50.0 keV'])

In [ ]:
df_list_extrafiltered = []
for loc, ind in enumerate(indices):
    if ind in indices_refined:
        df_filtered  = df_list[loc].copy()
        
        filtered_2550 = savgol_filter(df_list[loc]['25.0 keV-50.0 keV'], window_length = 305, polyorder=2)
        filtered_50100 = savgol_filter(df_list[loc]['50.0 keV-100.0 keV'], window_length = 305, polyorder=2)

        df_filtered['25.0 keV-50.0 keV'] = filtered_2550
        df_filtered['50.0 keV-100.0 keV'] = filtered_50100

        df_list_extrafiltered.append(df_filtered)
        
    if ind not in indices_refined:
        df_list_extrafiltered.append('excluded')

In [ ]:
df_list_extrafiltered[0].plot()

In [ ]:
i=0
peak_locs = [[] for _ in range(len(indices_refined))]
for ind in indices:
    if ind in indices_refined:
        fig, ax = plt.subplots()
        ax.set_ylabel('ct/(keV s)')
        ax.set_title(f"{parse_time(top100['start_UTC'].loc[ind]).datetime.year}-{parse_time(top100['start_UTC'].loc[ind]).datetime.month}")
        sci_df = df_list_extrafiltered[np.where(indices_array==ind)[0][0]]

        r = top100['solo_position_AU_distance'].loc[ind]
        
        tstart = filtered_stats_df['start_time'].iloc[i]
        tend = filtered_stats_df['end_time'].iloc[i]

        start = parse_time(top100['start_UTC'].loc[ind]).datetime - dt.timedelta(minutes=30)
        end = parse_time(top100['end_UTC'].loc[ind]).datetime  + dt.timedelta(minutes=30)
    
        plot_df = sci_df.truncate(start, end)
        plot_df.index = parse_time(plot_df.index).datetime 
        #plot_df = sci_df.truncate(t(tstart)-dt.timedelta(minutes=10), t(tend)+dt.timedelta(minutes=10))

        test_df = plot_df[np.abs(plot_df['25.0 keV-50.0 keV'].diff())>0.1]

        sci_df = df_list[np.where(indices_array==ind)[0][0]]
        plot_df = sci_df.truncate(start, end)
        
        ax.plot(plot_df.index, plot_df['25.0 keV-50.0 keV'])
        ax.plot(test_df.index, test_df['25.0 keV-50.0 keV'], 'o', ms=1)

        fig.savefig("C:/Users/derva/OneDrive/Documents/DIAS/solar-orbiter-flare-statistics/figures/week_4/difference_testing/flare"+str(i)+".png", bbox_inches='tight')

        i+=1
    
plt.close('all')

# tstart = filtered_stats_df['start_time'].iloc[0]
# tend = filtered_stats_df['end_time'].iloc[0]

# plot_df= df_list_filtered[0].truncate(t(tstart)-dt.timedelta(minutes=10), t(tend)+dt.timedelta(minutes=10))
# plt.plot(plot_df.index, plot_df['25.0 keV-50.0 keV'].diff())

In [ ]:
i=0
peak_locs = [[] for _ in range(len(indices_refined))]
for ind in indices:
    if ind in indices_refined:
        fig, ax = plt.subplots()
        ax.set_ylabel('ct/(keV s)')
        ax.set_title(f"{parse_time(top100['start_UTC'].loc[ind]).datetime.year}-{parse_time(top100['start_UTC'].loc[ind]).datetime.month}")
        sci_df = df_list_filtered[np.where(indices_array==ind)[0][0]]

        r = top100['solo_position_AU_distance'].loc[ind]
        
        tstart = filtered_stats_df['start_time'].iloc[i]
        tend = filtered_stats_df['end_time'].iloc[i]

        tstart_50_100 = filtered_stats_df['start_time (50-100 keV)'].iloc[i]
        tend_50_100 = filtered_stats_df['end_time (50-100 keV)'].iloc[i]

        # plt.axvline(t(tend), 0, 1, color = 'palevioletred', linestyle = 'dashed', label = 'Estimated Start/End (25-50 keV)')
        # plt.axvline(t(tstart), 0, 1, color = 'palevioletred', linestyle = 'dashed')
        # plt.axhline(10/r**2, 0, 1, color='red', linestyle='dotted', label='10 counts/keV s (scaled)')
        # plt.axhline(3/r**2, 0, 1, color='salmon', linestyle='dotted', label='3 counts/keV s (scaled)')

        # if pd.notna(tend_50_100):
        #     plt.axvline(t(tend_50_100), 0, 1, color = 'pink', linestyle = 'dashed', label = 'Estimated Start/End (50-100 keV)')
        #     plt.axvline(t(tstart_50_100), 0, 1, color = 'pink', linestyle = 'dashed')
        
        plot_df = sci_df.truncate(t(tstart)-dt.timedelta(minutes=10), t(tend)+dt.timedelta(minutes=10))

        peaks, properties = find_peaks(plot_df['25.0 keV-50.0 keV'], distance=15, prominence=0.1*peak_counts)

        # sci_df = df_list[np.where(indices_array==ind)[0][0]]
        # plot_df = sci_df.truncate(t(tstart)-dt.timedelta(minutes=10), t(tend)+dt.timedelta(minutes=10))
        for n in range(len(plot_df.columns)):
            ax.plot(plot_df.index, plot_df[plot_df.columns[n]], label=plot_df.columns[n])

        plt.scatter(plot_df.index[peaks], plot_df['25.0 keV-50.0 keV'].iloc[peaks], color='black', marker='*')
        
        handles, labels = ax.get_legend_handles_labels()
        fig.legend(handles, labels, loc='upper right', fontsize='small')
        fig.savefig("C:/Users/derva/OneDrive/Documents/DIAS/solar-orbiter-flare-statistics/figures/week_4/peak_testing/flare"+str(i)+".png", bbox_inches='tight')

        i+=1
    
plt.close('all')

In [ ]:
i=0
for ind in indices:
    if ind in indices_refined:
        fig, ax = plt.subplots()
        ax.set_ylabel('ct/(keV s)')
        ax.set_title(f"{parse_time(top100['start_UTC'].loc[ind]).datetime.year}-{parse_time(top100['start_UTC'].loc[ind]).datetime.month}")
        sci_df = df_list_filtered[np.where(indices_array==ind)[0][0]]
        
        start = parse_time(top100['start_UTC'].loc[ind]).datetime - dt.timedelta(minutes=30)
        end = parse_time(top100['end_UTC'].loc[ind]).datetime  + dt.timedelta(minutes=30)
        
        plot_df = sci_df.truncate(start, end)
             
        plot_df.index = t(plot_df.index)
        
        for n in range(len(plot_df.columns)):
            ax.plot(plot_df.index, plot_df[plot_df.columns[n]], label=plot_df.columns[n])
    
        # plt.axvline(parse_time(top100['peak_UTC'].loc[ind]).datetime, 0, 1, color = 'darkgrey', linestyle = 'dashed', label='Flare List Peak')
        # plt.axvline(plot_df.index[plot_df['25.0 keV-50.0 keV'] == np.max(plot_df['25.0 keV-50.0 keV'])][0], 0, 1, color = 'lightblue', linestyle = 'dashed', label='25-50 keV Peak')
        # plt.axvline(parse_time(top100['start_UTC'].loc[ind]).datetime, 0, 1, color = 'yellow', linestyle = 'dashed', label='Flare List Start/End')
        # plt.axvline(parse_time(top100['end_UTC'].loc[ind]).datetime, 0, 1, color = 'yellow', linestyle = 'dashed')

        # r = top100['solo_position_AU_distance'].loc[ind]
        
        # flare = plot_df[plot_df['25.0 keV-50.0 keV']>10/r**2]
        # tstart = flare.index[0]
        # tend = flare.index[-1]
    
        # if t(tend) in plot_df.index:
        #     plt.axvline(t(tend), 0, 1, color = 'pink', linestyle = 'dotted', label = 'Estimated Start/End')
        # if t(tstart) in plot_df.index:
        #     plt.axvline(t(tstart), 0, 1, color = 'pink', linestyle = 'dotted')
        
        #plt.axhline(10/r**2, 0, 1, color='red', linestyle='dotted')
        plt.yscale('log')
        
        handles, labels = ax.get_legend_handles_labels()
        fig.legend(handles, labels, loc='upper right', fontsize='small')
        fig.savefig("C:/Users/derva/OneDrive/Documents/DIAS/solar-orbiter-flare-statistics/figures/week_4/log_scale_flares/flare"+str(i)+".png", bbox_inches='tight')

        i+=1
    
plt.close('all')

In [ ]:
df_list_extrafiltered[0].shape

In [ ]:
bad_diff = [1, 7, 9, 23, 25, 26, 27, 33, 40, 47, 61, 73, 75, 35, 37, 41, 42, 44, 52, 53, 56, 58, 60, 63, 67, 70, 74, 76, 78, 80]

In [ ]:
durations_diff = []
estimated_starts = []
estimated_ends = []
i=0
for ind in indices:
    if ind in indices_refined:
        if i not in bad_diff:
            fig, ax = plt.subplots()
            ax.set_ylabel('ct/(keV s)')
            ax.set_title(f"{parse_time(top100['start_UTC'].loc[ind]).datetime.year}-{parse_time(top100['start_UTC'].loc[ind]).datetime.month}")
            sci_df = df_list_extrafiltered[np.where(indices_array==ind)[0][0]]

            start = parse_time(top100['start_UTC'].loc[ind]).datetime - dt.timedelta(minutes=30)
            end = parse_time(top100['end_UTC'].loc[ind]).datetime  + dt.timedelta(minutes=30)
        
            plot_df = sci_df.truncate(start, end)
            plot_df.index = parse_time(plot_df.index).datetime 
            #plot_df = sci_df.truncate(t(tstart)-dt.timedelta(minutes=10), t(tend)+dt.timedelta(minutes=10))
    
            test_df = plot_df[np.abs(plot_df['25.0 keV-50.0 keV'].diff())>0.05]
            diff_estimated_start = test_df.index[0]
            diff_estimated_end = test_df.index[-1]

            estimated_duration = (diff_estimated_end - diff_estimated_start).total_seconds()
            durations_diff.append(estimated_duration)
            
            plt.axvline(diff_estimated_start, 0, 1, color = 'pink', linestyle = 'dotted')
            plt.axvline(diff_estimated_end, 0, 1, color = 'pink', linestyle = 'dotted')
            
            sci_df = df_list[np.where(indices_array==ind)[0][0]]
            
            tstart = t(filtered_stats_df['start_time'].iloc[i]) - dt.timedelta(minutes=10)
            tend = t(filtered_stats_df['end_time'].iloc[i]) + dt.timedelta(minutes=10)
    
            plot_df = sci_df.truncate(start, end)
            
            ax.plot(plot_df.index, plot_df['25.0 keV-50.0 keV'])
            #ax.plot(test_df.index, test_df['25.0 keV-50.0 keV'], 'o', ms=1)
    
            fig.savefig("C:/Users/derva/OneDrive/Documents/DIAS/solar-orbiter-flare-statistics/figures/week_4/difference_testing/difference_good_flares/flare"+str(i)+".png", bbox_inches='tight')
        else:
            durations_diff.append(np.nan)
        i+=1
    
plt.close('all')

In [ ]:
plt.hist(durations_diff, bins=25)

In [ ]:
durations_test = []
for i in range(len(durations)):
    if i not in bad_diff:
        durations_test.append(durations[i])

In [ ]:
plt.hist(durations_test, bins=25)

In [ ]:
np.where(np.array(durations_diff)>3000)

In [ ]:
def gauss(x, A, x0, sigma):
    return A * np.exp(-(x - x0)**2 / (2 * sigma**2))

In [ ]:
# i=0
# peak_locs = [[] for _ in range(len(indices_refined))]
# for ind in indices:
#     if ind in indices_refined:
#         fig, ax = plt.subplots()
#         ax.set_ylabel('ct/(keV s)')
#         ax.set_title(f"{parse_time(top100['start_UTC'].loc[ind]).datetime.year}-{parse_time(top100['start_UTC'].loc[ind]).datetime.month}")
#         sci_df = df_list_extrafiltered[np.where(indices_array==ind)[0][0]]

#         sigma = (fwhm_list[i]/(2*np.sqrt(2*np.log(2)))).total_seconds()
#         A = filtered_stats_df['peak_counts'].iloc[i]
#         x0 = t(filtered_stats_df['peak_time'].iloc[i])
        
#         tstart = filtered_stats_df['start_time'].iloc[i]
#         tend = filtered_stats_df['end_time'].iloc[i]

#         tstart_50_100 = filtered_stats_df['start_time (50-100 keV)'].iloc[i]
#         tend_50_100 = filtered_stats_df['end_time (50-100 keV)'].iloc[i]

#         plot_df = sci_df.truncate(t(tstart)-dt.timedelta(minutes=10), t(tend)+dt.timedelta(minutes=10))
#         for n in range(len(plot_df.columns)):
#             ax.plot(plot_df.index, plot_df[plot_df.columns[n]], label=plot_df.columns[n])

#         ax.plot(plot_df.index, gauss(plot_df.index, A, x0, sigma))
        
#         handles, labels = ax.get_legend_handles_labels()
#         fig.legend(handles, labels, loc='upper right', fontsize='small')
#         fig.savefig("C:/Users/derva/OneDrive/Documents/DIAS/solar-orbiter-flare-statistics/figures/week_4/fwhm_testing/gaussian/flare"+str(i)+".png", bbox_inches='tight')

#         i+=1
    
# plt.close('all')